# 01 — Export LLaMA 3 to ONNX

Load a model config, export the decoder (with KV-cache) to ONNX via HF Optimum, and numerically verify the result against the PyTorch reference.

**GPU required** for the export cells. Start small with the tiny dev model on a free T4, then switch `MODEL` to the real 8B config on an A100.

In [ ]:
# ── Environment check ──────────────────────────────────
import sys
sys.path.insert(0, "/content/llm-inference-optimizer")  # Colab path
sys.path.insert(0, "..")                                  # local path fallback

from src.utils.env import get_env_info
info = get_env_info()
print(f"Device   : {info['device']}")
print(f"GPU      : {info['gpu_name']}")
print(f"CUDA     : {info['cuda_version']}")
print(f"PyTorch  : {info['torch_version']}")
print(f"Colab    : {info['is_colab']}")

## 1. Choose the model

`TINY` is an ungated tiny LLaMA used to validate the pipeline (no HF token needed). Set `USE_TINY = False` to export the real LLaMA 3 8B from `configs/model_configs/` (needs `HF_TOKEN` + A100).

In [ ]:
from src.utils.config import load_model_config

USE_TINY = True  # flip to False for the real 8B export

if USE_TINY:
    MODEL = "hf-internal-testing/tiny-random-LlamaForCausalLM"
    DTYPE = "fp32"
else:
    cfg = load_model_config("llama3_8b")
    MODEL = cfg["model_id"]
    DTYPE = cfg["dtype"]
    # Gated model: authenticate first.
    # import os; from huggingface_hub import login; login(os.environ["HF_TOKEN"])
print(f"Model: {MODEL}  dtype: {DTYPE}")

## 2. Export to ONNX

Uses `text-generation-with-past` (decoder + KV-cache) with dynamic batch/sequence axes.

In [ ]:
from src.export.onnx_exporter import export_to_onnx

OUTPUT_DIR = "results/onnx/tiny" if USE_TINY else "results/onnx/llama3_8b"
onnx_path = export_to_onnx(
    model_name_or_path=MODEL,
    output_path=OUTPUT_DIR,
    dtype=DTYPE,
    opset_version=17,
    verify_export=True,   # PyTorch-vs-ONNX logit parity check
)
print("Exported:", onnx_path)

## 3. Inspect the exported graph

In [ ]:
import onnx

model = onnx.load(onnx_path)
print("IR version:", model.ir_version)
print("Opset:", [(o.domain or 'ai.onnx', o.version) for o in model.opset_import])
print("Inputs:")
for i in model.graph.input:
    print("  ", i.name)
print("Outputs:")
for o in model.graph.output:
    print("  ", o.name)

In [ ]:
# ── Save results to GitHub ──────────────────────────────
import subprocess
NOTEBOOK_NAME = "01_export"
subprocess.run(["git", "add", "results/", "notebooks/"], check=True)
subprocess.run(["git", "commit", "-m", f"results: {NOTEBOOK_NAME} run"], check=True)
subprocess.run(["git", "push"], check=True)
print("Results pushed to GitHub.")